In [1]:
import multiprocessing
import queue
import threading

class DataProcessor:
    def __init__(self, num_processes=4):
        self.num_processes = num_processes
        self.in_queue = multiprocessing.Queue()
        self.out_queue = multiprocessing.Queue()
        self.pool = multiprocessing.Pool(self.num_processes)
        self.writer_thread = threading.Thread(target=self.write_data_to_file, args=())

    def start(self):
        self.writer_thread.start()

    def process_data(self, data):
        # Your data processing code
        result = ...
        return result

    def write_data_to_file(self):
        with open('output.txt', 'w') as f:
            while True:
                try:
                    result = self.out_queue.get(timeout=1)
                except queue.Empty:
                    if not any(p.is_alive() for p in self.pool._pool):
                        break
                    else:
                        continue
                f.write(str(result) + '\n')

    def feed_data(self, data):
        self.in_queue.put(data)

    def stop(self):
        self.pool.close()
        self.pool.join()
        self.out_queue.put(None)
        self.writer_thread.join()

    def worker(self):
        while True:
            data = self.in_queue.get()
            if data is None:
                break
            result = self.process_data(data)
            self.out_queue.put(result)

        self.out_queue.put(None)

        return

    def spawn_workers(self):
        for i in range(self.num_processes):
            self.pool.apply_async(self.worker)


In [2]:
dp = DataProcessor(num_processes=4)
dp.start()
dp.spawn_workers()
for data in range(1000):
    dp.feed_data(data)
dp.stop()

In [3]:
dp.in_queue.qsize()

1000

In [4]:
dp.out_queue.qsize()

0

# worked

In [15]:
import multiprocessing
import threading

class DataProcessor:
    def __init__(self, generator, process_data_func, filename):
        self.generator = generator
        self.process_data_func = process_data_func
        self.filename = filename
        self.pool = multiprocessing.Pool(4)
        self.queue = multiprocessing.Queue()

    def start_processing(self):
        # start the thread for writing to the file
        writer_thread = threading.Thread(target=self.write_to_file)
        writer_thread.start()

        # use imap_unordered to apply the processing function to each piece of data
        for result in self.pool.imap_unordered(self.process_data_func, self.generator):
            # put the result into the queue for the writer thread to process
            self.queue.put(result)

        # add None to the queue to signal the writer thread to exit
        self.queue.put(None)
        writer_thread.join()

    def write_to_file(self):
        with open(self.filename, 'w') as f:
            while True:
                item = self.queue.get()
                if item is None:
                    break
                f.write(str(item)+"\n")
                

In [16]:
generator = num_gene(10) # your generator that yields data sequentially
processor = DataProcessor(generator, process_data, 'output.txt')
processor.start_processing()